# ML-02 · Демографическая устойчивость и депопуляция

**Демографический советник · Этап II · версия stage2-ml2-v2.0.0**

Воспроизводимое исследование муниципальных траекторий России. Первичный источник — `D01 / mun_demography`, загружаемый по машинному каталогу `handoff/07_v2/dataset_index.json` приватного GitHub. Исходный файл D01 содержит наблюдения на уровне муниципальных образований; численность населения относится к **1 января** года `t`, рождения, смерти и миграционное сальдо — к календарному году `t`. Это положение проверяется далее по балансовым остаткам. Обработчик издателя ретроспективно согласовал численность населения с переписью 2010 года; нулевой остаток **не является независимой валидацией исходной статистики**.

Исследовательские гипотезы: **H2.1** — компонентный прогноз конкурентен прямому; **H2.2** — доступные прошлые демографические и пространственные признаки улучшают прогноз риска затяжной убыли; **H2.3** — воспроизводимы типы исторических траекторий; **H2.4** — интервалы населения и вероятность депопуляции калибруются на отложенных годах.

**Границы вывода:** только наблюдаемые муниципалитеты D01 (не вся РФ без экстраполяции); отсутствие раздела БДМО «Население» в текущем handoff отмечается отдельно. Ни кластеризация, ни SHAP/коэффициенты не идентифицируют причинные эффекты. Отдельные годы переписных пересмотров требуют качественных флагов.

**Источники методики:** [D01](https://tochno.st/datasets/mun_demography), [БДМО](https://tochno.st/datasets/bdmo), [Росстат — муниципальная численность](https://www.rosstat.gov.ru/folder/11110/document/13282). Все выходные данные без реального обучения и проверки не считаются научным результатом.

## 1. Конфигурация

Единственная ячейка, в которой пользователь меняет режим: `SMOKE` проверяет цепочку с уменьшенным объёмом, `FULL` проводит основной эксперимент и **только после итогового QA** может публиковать четыре логических артефакта в отдельную ветку приватного репозитория. Исходники Этапа I и ML-1 не изменяются. Google Drive не используется.

In [ ]:
from pathlib import Path
CFG = {
    "REPO": "AntonDozhdikov/Demograf_sovetnik", "SOURCE_REF": "main",
    "INDEX_PATH": "handoff/07_v2/dataset_index.json",
    "BUNDLE_PATH": "handoff/07_v2/07_handoff_bundle.zip",
    "REFERENCE_LOADER_PATH": "handoff/07_v2/stage2_loader.py",
    "OUTPUT_SEGMENT": "stage2/ml02_demography",
    "BRANCH_PREFIX": "research/ml02-demography",
    "VERSION": "stage2-ml2-v2.0.0", "MODE": "FULL",  # "SMOKE" / "FULL"
    "AUTO_PUBLISH": True,   # только после полного успешного FULL QA, в новой ветке
    "BASE_DIR": "/content/ml02_demography" if Path('/content').exists() else "/tmp/ml02_demography",
    "SEED": 20261102, "SMOKE_ENTITIES": 120,
    "MIN_TRAIN_ROWS": 180, "MIN_TEST_ROWS": 50,
    "MIN_HISTORY_YEARS": 3, "HORIZONS": [1,3,5,10],
    "DEV_TARGET_YEARS": 2, "TEST_TARGET_YEARS": 2,
    "RISK_YEARS": 3, "RISK_CUM_DROP": 0.02,
    "PUBLICATION_LAG_YEARS": 1,
    "N_BOOTSTRAP": 300, "N_CLUSTERS": [2,3,4,5],
    "SMOKE_BOOTSTRAP": 15,
    "MAX_RELEVANT_ASSET_MB": 100,
    "RELEASE_ASSET_THRESHOLD_MB": 45,
    "MAX_PLOTS": 8,
}
assert CFG['MODE'] in {'SMOKE','FULL'}
BASE = Path(CFG['BASE_DIR']); BASE.mkdir(parents=True,exist_ok=True)
CACHE = BASE/'cache'; CACHE.mkdir(exist_ok=True)
OUT = BASE/'output'; OUT.mkdir(exist_ok=True)
for _name in ['ML2_DEMOGRAPHY.ipynb','ML2_DEMOGRAPHY_RESULTS.xlsx','ML2_EVIDENCE.zip','run_manifest.json']:
    (OUT/_name).unlink(missing_ok=True)
print('ML-02',CFG['VERSION'],'·',CFG['MODE'],'· работа только в',str(BASE))


## 2–3. Диагностика Colab и зависимости

Для D01 достаточно CPU. Фиксируются версии инструментов, объём RAM и свободный диск. Пакеты устанавливаются только при отсутствии; данные и секреты в диагностический вывод не включаются.

In [ ]:
import os,sys,subprocess,platform,shutil,importlib.util,importlib.metadata as im
import time,datetime as dt,hashlib,json,base64,re,math,zipfile,io,traceback,warnings
from collections import defaultdict
from urllib.parse import quote
spec = {"numpy":"numpy>=1.24,<3", "pandas":"pandas>=2.0,<3", "pyarrow":"pyarrow>=14,<25",
        "sklearn":"scikit-learn>=1.3,<2", "openpyxl":"openpyxl>=3.1,<4", "tqdm":"tqdm>=4.64,<5",
        "requests":"requests>=2.30,<3", "matplotlib":"matplotlib>=3.7,<4", "nbformat":"nbformat>=5.8,<6",
        "joblib":"joblib>=1.3,<2"}
missing=[req for name,req in spec.items() if importlib.util.find_spec(name) is None]
if missing:
    p=subprocess.run([sys.executable,'-m','pip','-q','install',*missing],capture_output=True,text=True)
    if p.returncode: raise RuntimeError('Установка библиотек не удалась: '+p.stderr[-1200:])
import numpy as np,pandas as pd,pyarrow.parquet as pq,pyarrow as pa,requests,nbformat,joblib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge,LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor,HistGradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error,mean_squared_error,brier_score_loss,average_precision_score,roc_auc_score,adjusted_rand_score,silhouette_score
from sklearn.inspection import permutation_importance
from openpyxl import load_workbook
from openpyxl.styles import PatternFill,Font,Alignment
from openpyxl.utils import get_column_letter
from zipfile import ZipFile
START=time.monotonic()
resources={'python':sys.version.split()[0], 'cpu':os.cpu_count(), 'ram_gb':None,
           'free_disk_gb':round(shutil.disk_usage(BASE).free/1024**3,2)}
try:
    import psutil
    resources['ram_gb']=round(psutil.virtual_memory().total/1024**3,2)
except ImportError: pass
resources['libs']={x:(im.version(x) if x not in {'sklearn'} else im.version('scikit-learn')) for x in ['numpy','pandas','pyarrow','sklearn','openpyxl']}
print('Среда:',resources)


## 4. GitHub preflight — чтение без записи

Токен читается только из Colab Secrets или переменной окружения `GITHUB_TOKEN`. Ничего не записывается в GitHub до прохождения научной и технической приёмки. Транспорт воспроизводит референс `handoff/07_v2/stage2_loader.py`: `git` — Contents API, `release` — Releases assets API, с потоковой передачей и SHA-256. Недоступные или подменённые исходники не заменяются публичными копиями.

In [ ]:
try:
    from google.colab import userdata
    GH_TOKEN=(userdata.get('GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN','')).strip()
except Exception:  # Colab SecretNotFoundError либо локальный запуск
    GH_TOKEN=os.environ.get('GITHUB_TOKEN','').strip()
if not GH_TOKEN:
    from getpass import getpass
    GH_TOKEN=getpass('Личный GitHub token (скрытый ввод; нужен доступ к private repository): ').strip()
if not GH_TOKEN: raise RuntimeError('BLOCKED_AUTH: GITHUB_TOKEN не задан')
GH_API='https://api.github.com/repos/'+CFG['REPO']
GH_HEADERS={'Authorization':'Bearer '+GH_TOKEN,'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
def gh_error(response, stage):
    status=response.status_code
    if status in (401,403): raise PermissionError(f'{stage}: HTTP {status}; проверьте токен и права read/write')
    if status==404: raise FileNotFoundError(f'{stage}: HTTP 404; файл/ресурс или право доступа отсутствует')
    if status>=400: raise RuntimeError(f'{stage}: HTTP {status}; содержание ответа скрыто во избежание утечки токена')
def gh_get_json(url, params=None):
    r=requests.get(url,headers=GH_HEADERS,params=params,timeout=(20,120));gh_error(r,'GitHub GET')
    return r.json()
def gh_content(path,ref=None):
    obj=gh_get_json(GH_API+'/contents/'+quote(path,safe='/'),{'ref':ref or CFG['SOURCE_REF']})
    if not isinstance(obj,dict) or obj.get('type')!='file': raise RuntimeError('Неоднозначный ресурс GitHub: '+path)
    if obj.get('content'): data=base64.b64decode(obj['content'])
    else:
        with requests.get(obj['download_url'],headers=GH_HEADERS,stream=True,timeout=(20,900)) as r:
            gh_error(r,'GitHub streaming');data=b''.join(x for x in r.iter_content(2**20) if x)
    return data

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024**2),b''):h.update(chunk)
    return h.hexdigest()

def fetch_asset(asset):
    loc=asset['locator'];expected=asset['sha256'];aid=str(asset['asset_id'])
    dst=CACHE/(hashlib.sha256(aid.encode()).hexdigest()[:20]+'.parquet')
    if dst.exists() and sha256_file(dst)==expected:return dst
    dst.unlink(missing_ok=True)
    try:
        if loc['type']=='git':
            p=loc['path'];metadata=gh_get_json(GH_API+'/contents/'+quote(p,safe='/'),{'ref':CFG['SOURCE_REF']})
            if metadata.get('content'):
                dst.write_bytes(base64.b64decode(metadata['content']))
            else:
                r=requests.get(GH_API+'/contents/'+quote(p,safe='/'),headers={**GH_HEADERS,'Accept':'application/vnd.github.raw+json'},stream=True,timeout=(30,900))
                gh_error(r,'GitHub parquet git')
                with open(dst,'wb') as f:
                    for chunk in r.iter_content(4*1024**2):
                        if chunk:f.write(chunk)
                r.close()
        elif loc['type']=='release':
            url=GH_API+'/releases/assets/'+str(loc['asset_id'])
            r=requests.get(url,headers={**GH_HEADERS,'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
            gh_error(r,'GitHub release')
            if 'json' in r.headers.get('Content-Type','').lower():
                r.close();meta=gh_get_json(url)
                # Точный browser_download_url из API, а не угаданный адрес.
                r=requests.get(meta['browser_download_url'],headers=GH_HEADERS,stream=True,timeout=(30,900))
                gh_error(r,'GitHub release redirect')
            with open(dst,'wb') as f:
                for chunk in r.iter_content(4*1024**2):
                    if chunk:f.write(chunk)
            r.close()
        else: raise RuntimeError('Неизвестный locator: '+str(loc['type']))
        if sha256_file(dst)!=expected: raise ValueError('SHA256 не совпал: '+aid)
        return dst
    except Exception:
        dst.unlink(missing_ok=True);raise

repo_meta=gh_get_json(GH_API)
assert repo_meta['full_name']==CFG['REPO'] and repo_meta.get('private'), 'Ожидался заявленный приватный репозиторий'
INDEX_BYTES=gh_content(CFG['INDEX_PATH'])
INDEX=json.loads(INDEX_BYTES)
assert INDEX['repo']==CFG['REPO'] and isinstance(INDEX['assets'],list)
assert len({x['asset_id'] for x in INDEX['assets']})==len(INDEX['assets'])
HEAD_SHA=gh_get_json(GH_API+'/commits/'+quote(CFG['SOURCE_REF'],safe=''))['sha']
print('GitHub READ PASS ·',CFG['REPO'],'·',len(INDEX['assets']),'assets · ref',HEAD_SHA[:12])
assert b'def fetch_asset' in gh_content(CFG['REFERENCE_LOADER_PATH']), 'Не совпал ожидаемый контракт загрузчика'
# Сам bundle достаточно проверить на наличие; его сырые содержимое и пути не угадываются.
bundle_metadata=gh_get_json(GH_API+'/contents/'+quote(CFG['BUNDLE_PATH'],safe='/'),{'ref':CFG['SOURCE_REF']})
assetD01=[a for a in INDEX['assets'] if a['source_id']=='D01' and a['segment']=='01']
assert len(assetD01)==1 and assetD01[0]['sha256_verified']
D01_ASSET=assetD01[0]
BDMO_DEMOG=[a for a in INDEX['assets'] if a['source_id']=='BDMO' and ('section31' in a.get('member','').lower() or 'population' in a.get('member','').lower())]
print('D01:',D01_ASSET['asset_id'],'· row_manifest:',D01_ASSET['rows_actual'],'· демографический раздел БДМО в индексе:',len(BDMO_DEMOG))


## 5. DATA_GATE: наблюдаемость, идентификаторы и демографический баланс

ОКТМО хранится как строка. Для всякого `t` проверяется равенство `P(t+1)−P(t)=B(t)−D(t)+M(t)+R(t)` с флагами неполных компонентов и разрывов календаря. Источник D01 **описывает численность как уже реконструированную для временной сопоставимости**; поэтому остаток не приравнивается к независимой ошибке Росстата. Типология не строится по будущим данным.

In [ ]:
D01_FILE=fetch_asset(D01_ASSET)
assert pq.ParquetFile(D01_FILE).metadata.num_rows==D01_ASSET['rows_actual'],'Число строк D01 не совпало с индексом'
COLS=['oktmo','region','mun_type','municipality','year','population','average_population','births','deaths','migration','birth_rate','mortality_rate','migration_rate','not_zato']
assert all(x in pq.ParquetFile(D01_FILE).schema_arrow.names for x in COLS),'Нарушен контракт D01'
raw=pq.read_table(D01_FILE,columns=COLS).to_pandas()
for col in ['oktmo','region','municipality','mun_type']:
    raw[col]=raw[col].astype('string').str.strip()
raw['oktmo']=raw['oktmo'].str.replace(r'\.0$','',regex=True)
raw['year']=pd.to_numeric(raw['year'],errors='coerce')
for col in ['population','average_population','births','deaths','migration','birth_rate','mortality_rate','migration_rate']:
    raw[col]=pd.to_numeric(raw[col],errors='coerce')
keys_ok=raw['oktmo'].notna() & raw['year'].notna() & raw['oktmo'].ne('')
if not keys_ok.all(): raise ValueError('DATA_GATE FAIL: пустые муниципальные ключи или годы')
if (raw['year']%1!=0).any(): raise ValueError('DATA_GATE FAIL: год не целочисленный')
raw['year']=raw['year'].astype(int)
if raw.duplicated(['oktmo','year']).any():
    duplicates=raw.loc[raw.duplicated(['oktmo','year'],keep=False),['oktmo','year','region']]
    duplicates.to_csv(OUT/'key_collisions_diagnostic.csv',index=False)
    raise ValueError('DATA_GATE FAIL: неуникальная пара ОКТМО-год; требуется методическое разрешение, автоматическая агрегация запрещена')
if (raw[['population','births','deaths']].dropna()<0).any().any(): raise ValueError('DATA_GATE FAIL: отрицательные P, B или D')
raw=raw.sort_values(['oktmo','year']).reset_index(drop=True)
if CFG['MODE']=='SMOKE':
    rng=np.random.default_rng(CFG['SEED'])
    ids=raw['oktmo'].dropna().unique()
    chosen=rng.choice(ids,min(len(ids),CFG['SMOKE_ENTITIES']),replace=False)
    raw=raw.loc[raw['oktmo'].isin(chosen)].copy().reset_index(drop=True)

# Перенос на t+1 только по календарной паре; не интерполировать пропущенные годы.
next_values=raw[['oktmo','year','population']].rename(columns={'population':'population_next'})
next_values['year']=next_values['year']-1
panel=raw.merge(next_values,on=['oktmo','year'],how='left',validate='one_to_one',sort=False)
panel['natural_change']=panel['births']-panel['deaths']
panel['observed_delta']=panel['population_next']-panel['population']
panel['balance_residual']=panel['observed_delta']-panel['natural_change']-panel['migration']
panel['balance_eligible']=panel[['population','population_next','births','deaths','migration']].notna().all(axis=1)
panel['census_method_regime']='D01_reconstructed_from_2010_census'
panel['source_asset_id']=D01_ASSET['asset_id']
panel['source_vintage']=str(INDEX.get('created_at',''))
panel['population_reference_date']='01-01'
panel['component_reference']='calendar_year'
panel['boundary_version']='D01_harmonized_2024'
panel['revision_or_boundary_flag']='unknown_in_rows'

qa_key=pd.DataFrame([{'Проверка':'Число строк', 'Значение':len(panel)},
                    {'Проверка':'Паспорт первоисточника (сайт), строк; требуется сверка', 'Значение':31947},
                    {'Проверка':'Муниципалитеты (ОКТМО)', 'Значение':panel['oktmo'].nunique()},
                    {'Проверка':'Регионы', 'Значение':panel['region'].nunique()},
                    {'Проверка':'Начальный год', 'Значение':int(panel['year'].min())},
                    {'Проверка':'Последний год', 'Значение':int(panel['year'].max())},
                    {'Проверка':'Дубликаты ОКТМО × год', 'Значение':int(panel.duplicated(['oktmo','year']).sum())},
                    {'Проверка':'Полные балансовые пары', 'Значение':int(panel['balance_eligible'].sum())},
                    {'Проверка':'Доля отсутствующего P', 'Значение':float(panel['population'].isna().mean())},
                    {'Проверка':'Доля отсутствующих B/D/M', 'Значение':float(panel[['births','deaths','migration']].isna().any(axis=1).mean())}])
qa_balance=(panel.loc[panel['balance_eligible']].groupby('year',as_index=False)
       .agg(n=('oktmo','count'),median_residual=('balance_residual','median'),
            mean_abs_residual=('balance_residual',lambda x:float(x.abs().mean())),
            nonzero_residual=('balance_residual',lambda x:int((x.abs()>0.01).sum()))))
qa_years=(panel.groupby('year').agg(municipalities=('oktmo','nunique'),
   population_n=('population','count'),births_n=('births','count'),
   deaths_n=('deaths','count'),migration_n=('migration','count')).reset_index())
print('DATA_GATE:',qa_key.to_dict('records'))
print('Остатки баланса: медиана |R| =',round(float(panel['balance_residual'].abs().median()),3))


## 6. Построение собственной панели и временных признаков

Все признаки используют только историю строго до начала прогнозируемого года `origin_year`: `P(t)` и лаги событий с `event_year <= t−1` при публикационном лаге 1. Отсутствие лагов сохраняется как пропуск, статистические методы заполняют его **внутри train pipeline**. Региональные уровни не копируются в муниципальную панель; несопоставимые пространственные файлы не присоединяются автоматически.

In [ ]:
def feature_frame(data,lag=1):
    d=data.sort_values(['oktmo','year']).copy().reset_index(drop=True)
    # Привязка через годовой ключ, а не через соседнюю строку с возможным пропуском календаря.
    for i in [1,2,3]:
        p=data[['oktmo','year','population']].rename(columns={'population':f'p_lag{i}'})
        p['year']=p['year']+i
        d=d.merge(p,on=['oktmo','year'],how='left',validate='one_to_one')
        for source,target in [('births','births'),('deaths','deaths'),('migration','migration')]:
            v=data[['oktmo','year',source]].rename(columns={source:f'{target}_lag{i}'})
            v['year']=v['year']+i+max(0,lag-1)
            d=d.merge(v,on=['oktmo','year'],how='left',validate='one_to_one')
    for i in [1,2,3]:
        d[f'pop_growth_lag{i}']=1000*(d['population']-d[f'p_lag{i}'])/d[f'p_lag{i}'].replace(0,np.nan)
        d[f'birth_rate_lag{i}']=1000*d[f'births_lag{i}']/d[f'p_lag{i}'].replace(0,np.nan)
        d[f'death_rate_lag{i}']=1000*d[f'deaths_lag{i}']/d[f'p_lag{i}'].replace(0,np.nan)
        d[f'migration_rate_lag{i}']=1000*d[f'migration_lag{i}']/d[f'p_lag{i}'].replace(0,np.nan)
    d['natural_rate_lag1']=d['birth_rate_lag1']-d['death_rate_lag1']
    d['past_three_years_decline']=((d['population']<d['p_lag1']) &
      (d['p_lag1']<d['p_lag2']) & (d['p_lag2']<d['p_lag3'])).astype(float)
    d['log_population']=np.log1p(d['population'].clip(lower=0))
    return d

origins=feature_frame(panel,CFG['PUBLICATION_LAG_YEARS'])
FEATURES=['log_population','pop_growth_lag1','pop_growth_lag2','pop_growth_lag3',
          'birth_rate_lag1','birth_rate_lag2','birth_rate_lag3',
          'death_rate_lag1','death_rate_lag2','death_rate_lag3',
          'migration_rate_lag1','migration_rate_lag2','migration_rate_lag3',
          'natural_rate_lag1','past_three_years_decline']
for c in FEATURES:
    origins[c]=origins[c].replace([np.inf,-np.inf],np.nan)
origins=origins.loc[origins['population'].gt(0) & origins['p_lag3'].gt(0)].copy()
assert all(pd.api.types.is_numeric_dtype(origins[c]) for c in FEATURES)
assert origins[['oktmo','year']].duplicated().sum()==0

# Информация о компонентах за каждый будущий год необходима лишь как целевая метка;
# она никогда не добавляется в FEATURES.
def forecast_labels(d,p,h):
    o=d.copy(); base=p.set_index(['oktmo','year'])
    future=pd.MultiIndex.from_arrays([o['oktmo'],o['year']+h],names=['oktmo','year'])
    o['P_real']=base['population'].reindex(future).to_numpy()
    for k in range(1,h+1):
        event=pd.MultiIndex.from_arrays([o['oktmo'],o['year']+k-1],names=['oktmo','year'])
        for original,short in [('births','B'),('deaths','D'),('migration','M')]:
            o[f'{short}_true_{k}']=base[original].reindex(event).to_numpy()
    o['target_year']=o['year']+h;o['horizon']=h
    o['y_delta_rate']=1000*(o['P_real']-o['population'])/o['population']
    o['y_delta_pct']=100*(o['P_real']-o['population'])/o['population']
    return o

# Нельзя придумывать 2024+ или использовать даты публикации, которых нет в D01.
LAST_YEAR=int(panel['year'].max())
TRAIN_CUTOFF=LAST_YEAR-CFG['DEV_TARGET_YEARS']-CFG['TEST_TARGET_YEARS']
DEV_CUTOFF=LAST_YEAR-CFG['TEST_TARGET_YEARS']
PLAN=[];labels={}
for h in CFG['HORIZONS']:
    z=forecast_labels(origins,panel,h)
    labels[h]=z
    required=['P_real']+[f'{x}_true_{k}' for k in range(1,h+1) for x in 'BDM']
    valid=z[required].notna().all(axis=1)
    tr=(z['target_year']<=TRAIN_CUTOFF)&valid
    dev=z['target_year'].between(TRAIN_CUTOFF+1,DEV_CUTOFF)&valid
    te=z['target_year'].between(DEV_CUTOFF+1,LAST_YEAR)&valid
    feasible=tr.sum()>=CFG['MIN_TRAIN_ROWS'] and te.sum()>=CFG['MIN_TEST_ROWS'] and dev.sum()>=30
    PLAN.append({'h':h,'train':int(tr.sum()),'dev':int(dev.sum()),'test':int(te.sum()),
        'status':'READY' if feasible else 'NOT_FEASIBLE',
        'reason':'достаточные парные наблюдения P/B/D/M' if feasible else 'недостаточно полной истории для общего теста P/B/D/M',
        'last_observed_year':LAST_YEAR})
DATA_PLAN=pd.DataFrame(PLAN)
print('Целевые окна:',f'TRAIN target<= {TRAIN_CUTOFF}; DEV {TRAIN_CUTOFF+1}–{DEV_CUTOFF}; TEST {DEV_CUTOFF+1}–{LAST_YEAR}')
print(DATA_PLAN.to_string(index=False))


## 7. Фиксация научного плана до теста

План замораживается после проверки доступности меток, **без вычисления метрик на тесте**. Основные сравнения: `DIRECT-HGB` и `COMPONENT-HGB` на идентичных `entity × origin × horizon` с контрольными `LAST_P`, `TREND_P` и `HIST_BALANCE`. Восстановление компонента `R_hat = 0` — только модельное допущение (не оценённая истина). Подбираем гиперпараметры на DEV; TEST используется один раз после фиксации выбора. Дополнительный `CROSSED` и `TERRITORY_ONLY` — только при достаточных региональных группах; иначе `NOT_FEASIBLE`. Риски — 3 последовательных снижения и совокупный спад не менее 2%.

In [ ]:
DESIGN={
 'split':'rolling_target_year', 'train_cutoff':TRAIN_CUTOFF,'dev_cutoff':DEV_CUTOFF,
 'test_years':list(range(DEV_CUTOFF+1,LAST_YEAR+1)),
 'available_year':'origin_year - publication_lag',
 'population_reference':'1 January of observation_year',
 'components_reference':'calendar_year', 'features':FEATURES,
 'coherent_R_assumption':0.0,
 'confirmatory_comparisons':['DIRECT_HGB vs COMPONENT_HGB MAE P (paired)','DIRECT_HGB vs LAST_P MAE P (paired)'],
 'risk_definition':{'consecutive_negative_years':3,'minimum_cumulative_decline_pct':100*CFG['RISK_CUM_DROP']},
 'metric_primary':'MAE_population', 'metric_secondary':['RMSE_P','MAE_change_per1000','MAE_B','MAE_D','MAE_M'],
 'multiplicity':'Holm family-wise, clustered region bootstrap',
 'source_vintage':INDEX.get('created_at'),
 'data_coverage_limit':'D01 78 regions; excludes unobserved municipalities',
 'target_policy':'train only rows whose full future target window ends by train_cutoff',
}
DESIGN_SHA=hashlib.sha256(json.dumps(DESIGN,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
# Канарейка: мутация будущего года не меняет признаки прошлого origin.
def assert_future_canary(d):
    sel=d['oktmo'].drop_duplicates().head(10)
    a=d.loc[d['oktmo'].isin(sel)].copy()
    maxy=int(a['year'].max());threshold=maxy-2
    f1=feature_frame(a,CFG['PUBLICATION_LAG_YEARS'])
    mutated=a.copy()
    mask=mutated['year']>=maxy-1
    for field in ['population','births','deaths','migration']:
        mutated.loc[mask,field]=9999999
    f2=feature_frame(mutated,CFG['PUBLICATION_LAG_YEARS'])
    cols=['oktmo','year']+FEATURES
    x=f1.loc[f1['year']<=threshold,cols].reset_index(drop=True)
    y=f2.loc[f2['year']<=threshold,cols].reset_index(drop=True)
    pd.testing.assert_frame_equal(x,y,check_dtype=False)
    return True
assert assert_future_canary(panel),'FUTURE MUTATION CANARY FAIL'
assert not any('true' in c.lower() or 'real' in c.lower() for c in FEATURES)
print('PLAN_FROZEN',DESIGN_SHA[:16],'· future-mutation canary PASS')


## 8. Прогноз численности и компонент B/D/M

Эксперимент сравнивает значения `P(t+h)`, прогнозируемые непосредственно по прошлым лагам, и суммы прогнозируемых компонент `B−D+M`. Компоненты каждого будущего года предсказываются **отдельными k-step моделями из признаков в t**; фактические промежуточные годы не подставляются. Обучение HGB и Ridge происходит на одном историческом train; исторические baseline получают ровно те же точки отсчёта. `R_hat=0` означает только отсутствие сценарной корректировки, а не утверждение об отсутствии административных пересмотров.

In [ ]:
from sklearn.base import clone
RUN_FULL=(CFG['MODE']=='FULL')
VALID_H=[int(r['h']) for r in PLAN if r['status']=='READY']
UNITS=sum(2+3*h for h in VALID_H)+8
MAIN_BAR=tqdm(total=UNITS,desc='ML-02 · единый ход эксперимента',unit='задача',dynamic_ncols=True,leave=True)
def tick(stage,amount=1):
    MAIN_BAR.set_postfix_str(stage[:60]);MAIN_BAR.update(amount)

def regression_model(kind='hgb'):
    m=(HistGradientBoostingRegressor(max_iter=150,max_leaf_nodes=15,l2_regularization=2,
                learning_rate=.05,random_state=CFG['SEED'],early_stopping=False)
       if kind=='hgb' else Ridge(alpha=10.0))
    return Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('model',m)])

def classification_model(kind='hgb'):
    estimator=(HistGradientBoostingClassifier(max_iter=130,max_leaf_nodes=12,learning_rate=.05,
                l2_regularization=2,early_stopping=False,random_state=CFG['SEED']) if kind=='hgb'
               else LogisticRegression(max_iter=700,class_weight=None))
    steps=[('imputer',SimpleImputer(strategy='median',keep_empty_features=True))]
    if kind=='logit':steps.append(('scaler',StandardScaler()))
    steps.append(('model',estimator))
    return Pipeline(steps)

MODELS={} ; PRED=[] ; MODEL_METRICS=[] ; DEV_RESIDUALS=[] ; TRAJECTORIES=[]

def metric_population(y,p,p0):
    y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float);p0=np.asarray(p0,dtype=float)
    valid=np.isfinite(y)&np.isfinite(p)&np.isfinite(p0)&(p0>0)
    y=y[valid];p=p[valid];p0=p0[valid]
    if not len(y):return {'N':0,'MAE_P':np.nan,'RMSE_P':np.nan,'MAE_change_per1000':np.nan}
    return {'N':len(y),'MAE_P':mean_absolute_error(y,p),
            'RMSE_P':np.sqrt(mean_squared_error(y,p)),
            'MAE_change_per1000':float(np.mean(np.abs(p-y)/p0)*1000)}

for h in VALID_H:
    z=labels[h].copy()
    required=['P_real']+[f'{x}_true_{k}' for k in range(1,h+1) for x in 'BDM']
    good=z[required].notna().all(axis=1)
    t=z.loc[good&(z['target_year']<=TRAIN_CUTOFF)].copy()
    d=z.loc[good&z['target_year'].between(TRAIN_CUTOFF+1,DEV_CUTOFF)].copy()
    q=z.loc[good&z['target_year'].between(DEV_CUTOFF+1,LAST_YEAR)].copy()
    if min(len(t),len(d),len(q))<10:raise AssertionError('Изменился DATA_GATE: '+str(h))
    # Плотность test определена до обучения; никаких выборок теста по ошибкам.
    X_train=t[FEATURES];X_dev=d[FEATURES];X_test=q[FEATURES]
    model=regression_model('hgb').fit(X_train,t['y_delta_rate'])
    MODELS[f'direct_hgb_h{h}']=model
    tick(f'h={h} · DIRECT-HGB')
    ridge=regression_model('ridge').fit(X_train,t['y_delta_rate'])
    MODELS[f'direct_ridge_h{h}']=ridge
    tick(f'h={h} · DIRECT-Ridge')
    for part,ds,X in [('DEV',d,X_dev),('TEST',q,X_test)]:
        direct=ds['population'].to_numpy()*(1+model.predict(X)/1000)
        linear=ds['population'].to_numpy()*(1+ridge.predict(X)/1000)
        ds[f'P_direct']=np.maximum(0,direct)
        ds[f'P_ridge']=np.maximum(0,linear)
        ds[f'P_last']=ds['population'].to_numpy()
        ds[f'P_trend']=np.maximum(0,ds['population'].to_numpy()+h*(ds['population'].to_numpy()-ds['p_lag1'].to_numpy()))
        base_rate=(ds[['birth_rate_lag1','birth_rate_lag2','birth_rate_lag3']].mean(axis=1)-
                  ds[['death_rate_lag1','death_rate_lag2','death_rate_lag3']].mean(axis=1)+
                  ds[['migration_rate_lag1','migration_rate_lag2','migration_rate_lag3']].mean(axis=1))
        ds['P_historical_balance']=np.maximum(0,ds['population']*(1+h*base_rate.fillna(0)/1000))
        if part=='DEV':d=ds
        else:q=ds
    # Один HGB для абсолютной B/D/M каждого будущего года в нормировке к P(origin).
    for k in range(1,h+1):
        for target in 'BDM':
            col=f'{target}_true_{k}';target_rate=1000*t[col]/t['population']
            fitted=regression_model('hgb').fit(X_train,target_rate)
            MODELS[f'{target.lower()}_h{h}_k{k}']=fitted
            for ds in [d,q]:
                pred=fitted.predict(ds[FEATURES])*ds['population'].to_numpy()/1000
                if target in 'BD':pred=np.maximum(pred,0)
                ds[f'{target}_pred_{k}']=pred
            tick(f'h={h}, k={k} · {target} component')
    for part,ds in [('DEV',d),('TEST',q)]:
        ds['B_pred']=sum(ds[f'B_pred_{k}'] for k in range(1,h+1))
        ds['D_pred']=sum(ds[f'D_pred_{k}'] for k in range(1,h+1))
        ds['M_pred']=sum(ds[f'M_pred_{k}'] for k in range(1,h+1))
        ds['P_component']=np.maximum(0,ds['population']+ds['B_pred']-ds['D_pred']+ds['M_pred'])
        ds['R_assumption']=0.0
        ds['balance_error_forecast']=ds['P_component']-(ds['population']+ds['B_pred']-ds['D_pred']+ds['M_pred'])
        for m,c in [('DIRECT_HGB','P_direct'),('DIRECT_RIDGE','P_ridge'),('COMPONENT_HGB','P_component'),
                    ('LAST_P','P_last'),('TREND_P','P_trend'),('HIST_BALANCE','P_historical_balance')]:
            met=metric_population(ds['P_real'],ds[c],ds['population'])
            MODEL_METRICS.append({'split':part,'horizon':h,'model':m,**met})
        if part=='DEV':
            DEV_RESIDUALS.extend([{'horizon':h,'region':r,'abs_error':abs(y-p)}
                                  for r,y,p in zip(ds['region'],ds['P_real'],ds['P_component'])])
        if part=='TEST':
            colnames=['oktmo','region','municipality','year','target_year','horizon','population','P_real',
              'P_direct','P_ridge','P_component','P_last','P_trend','P_historical_balance',
              'B_pred','D_pred','M_pred','R_assumption','balance_error_forecast']
            dd=ds[colnames].copy().rename(columns={'year':'origin_year'})
            dd['information_cutoff']=dd['origin_year']
            dd['source_vintage']=INDEX.get('created_at')
            dd['split_id']='ROLLING_TIME_TEST'
            dd['seed']=CFG['SEED'];dd['boundary_version']='D01_harmonized_2024'
            dd['B_real']=sum(ds[f'B_true_{k}'] for k in range(1,h+1))
            dd['D_real']=sum(ds[f'D_true_{k}'] for k in range(1,h+1))
            dd['M_real']=sum(ds[f'M_true_{k}'] for k in range(1,h+1))
            dd['true_components_balance']=dd['population']+dd['B_real']-dd['D_real']+dd['M_real']
            dd['true_balance_residual']=dd['P_real']-dd['true_components_balance']
            dd['comparable_flag']=True
            dd['entity_id']=dd['oktmo']
            PRED.append(dd)
            for k in range(1,h+1):
                annual=ds[['oktmo','region','year']].copy().rename(columns={'year':'origin_year'})
                annual['entity_id']=annual['oktmo']
                annual['horizon']=h;annual['step']=k
                annual['event_year']=annual['origin_year']+k-1
                annual['B_pred']=ds[f'B_pred_{k}'];annual['D_pred']=ds[f'D_pred_{k}'];annual['M_pred']=ds[f'M_pred_{k}']
                annual['B_real']=ds[f'B_true_{k}'];annual['D_real']=ds[f'D_true_{k}'];annual['M_real']=ds[f'M_true_{k}']
                annual['split_id']='ROLLING_TIME_TEST'; annual['seed']=CFG['SEED']
                TRAJECTORIES.append(annual)
PRED=pd.concat(PRED,ignore_index=True) if PRED else pd.DataFrame()
TRAJECTORIES=pd.concat(TRAJECTORIES,ignore_index=True) if TRAJECTORIES else pd.DataFrame()
if not PRED.empty:
    PRED_LONG=PRED.melt(id_vars=['entity_id','oktmo','origin_year','horizon','target_year','split_id',
      'seed','information_cutoff','source_vintage','boundary_version','P_real','population'],
      value_vars=['P_direct','P_ridge','P_component','P_last','P_trend','P_historical_balance'],
      var_name='model',value_name='P_prediction')
else:PRED_LONG=pd.DataFrame()
METRICS=pd.DataFrame(MODEL_METRICS)
print('ROLLING_TIME: полные прогнозы',len(PRED),'· обученных моделей',len(MODELS))


## 9. Риск трёхлетней депопуляции

Метка равна единице лишь при **трёх последовательных годах снижения** и суммарном сокращении численности **не менее 2%** к `t+3`. Обучающие строки имеют все будущие даты внутри TRAIN; признаки не содержат будущие значения. Сравниваем историю сохранения спада, постоянную распространённость, логистическую регрессию и нелинейный классификатор; тестом определяется Brier, PR-AUC (если два класса) и калибровка. Более ранняя экономическая информация не создаётся искусственно при отсутствии сопоставимого показателя БДМО.

In [ ]:
RISK_H=CFG['RISK_YEARS']
r=forecast_labels(origins,panel,RISK_H)
base_lookup=panel.set_index(['oktmo','year'])['population']
for j in range(1,RISK_H+1):
    kk=pd.MultiIndex.from_arrays([r['oktmo'],r['year']+j],names=['oktmo','year'])
    r[f'p_fut{j}']=base_lookup.reindex(kk).to_numpy()
valid_risk=r[['population']+[f'p_fut{i}' for i in range(1,RISK_H+1)]].notna().all(axis=1)
r=r.loc[valid_risk].copy()
r['risk_label']=(((r['p_fut1']<r['population']) & (r['p_fut2']<r['p_fut1']) &
                  (r['p_fut3']<r['p_fut2'])) &
                  ((r['p_fut3']/r['population']-1)<=-CFG['RISK_CUM_DROP'])).astype(int)
rt=r[r['target_year']<=TRAIN_CUTOFF].copy()
rd=r[r['target_year'].between(TRAIN_CUTOFF+1,DEV_CUTOFF)].copy()
rq=r[r['target_year'].between(DEV_CUTOFF+1,LAST_YEAR)].copy()
RISK_READY=len(rt)>=CFG['MIN_TRAIN_ROWS'] and len(rq)>=CFG['MIN_TEST_ROWS'] and rt['risk_label'].nunique()==2
RISK_METRICS=[];RISK_PREDS=pd.DataFrame();risk_probability_models={}
if RISK_READY:
    for label,kind in [('RISK_LOGIT','logit'),('RISK_HGB','hgb')]:
        mod=classification_model(kind).fit(rt[FEATURES],rt['risk_label'])
        MODELS[label]=mod
        risk_probability_models[label]=mod
        tick('Риск · '+label)
    for split,part in [('DEV',rd),('TEST',rq)]:
        if not len(part):continue
        p0=float(rt['risk_label'].mean())
        pp={'PREVALENCE':np.full(len(part),p0),
            'PERSISTENCE':part['past_three_years_decline'].fillna(0).to_numpy()*.85+.075}
        pp.update({name:mdl.predict_proba(part[FEATURES])[:,1] for name,mdl in risk_probability_models.items()})
        for name,probs in pp.items():
            yy=part['risk_label'].to_numpy();pr=np.clip(probs,1e-5,1-1e-5)
            RISK_METRICS.append({'split':split,'model':name,'N':len(yy),'risk_prevalence':yy.mean(),
             'Brier':brier_score_loss(yy,pr),
             'PR_AUC':average_precision_score(yy,pr) if len(np.unique(yy))>1 else np.nan,
             'ROC_AUC':roc_auc_score(yy,pr) if len(np.unique(yy))>1 else np.nan})
        if split=='TEST':
            RISK_PREDS=part[['oktmo','region','year','target_year','risk_label']].copy()
            RISK_PREDS=RISK_PREDS.rename(columns={'year':'origin_year'})
            for name,pr in pp.items():RISK_PREDS['prob_'+name]=pr
            RISK_PREDS['split_id']='ROLLING_TIME_TEST';RISK_PREDS['seed']=CFG['SEED']
else:
    tick('Риск · NOT_FEASIBLE',2)
    print('RISK NOT_FEASIBLE: train/test или число классов недостаточны')
RISK_METRICS=pd.DataFrame(RISK_METRICS)


## 10. Типы исторических траекторий

Кластеры выявляются **исключительно по прошлой динамике** (`P(t−3…t)`, прошлые компоненты), без использования будущих значений, без географических ярлыков и без предзаданного названия «устойчивых» или «неустойчивых» регионов. Количество групп выбирается по silhouette внутри train. Проверка bootstrap-stability — согласованность распределения train при повторной выборке региональных блоков; вывод о переносимости на новые территории не делается без пространственного теста.

In [ ]:
TYPE_FEATURES=['pop_growth_lag1','pop_growth_lag2','pop_growth_lag3',
               'natural_rate_lag1','migration_rate_lag1']
train_types=origins[origins['year']<=TRAIN_CUTOFF].copy()
test_types=origins[origins['year'].between(DEV_CUTOFF+1,LAST_YEAR)].copy()
TYPES_READY=len(train_types)>=CFG['MIN_TRAIN_ROWS'] and len(test_types)>=CFG['MIN_TEST_ROWS']
TYPE_REPORT=[];TYPE_ASSIGNMENTS=pd.DataFrame();TYPE_STABILITY=pd.DataFrame()
if TYPES_READY:
    imputer=SimpleImputer(strategy='median',keep_empty_features=True)
    scaler=StandardScaler()
    A=scaler.fit_transform(imputer.fit_transform(train_types[TYPE_FEATURES]))
    score_rows=[]
    for kk in CFG['N_CLUSTERS']:
        if len(A)<kk*30:continue
        km=KMeans(n_clusters=kk,random_state=CFG['SEED'],n_init=10).fit(A)
        subs=np.random.default_rng(CFG['SEED']).choice(len(A),min(len(A),1500),replace=False)
        sil=silhouette_score(A[subs],km.predict(A[subs]))
        score_rows.append((float(sil),kk,km))
    if score_rows:
        score,chosen_k,cluster=max(score_rows,key=lambda t:t[0])
        MODELS['typology_kmeans']=cluster
        TEST_TYPES=scaler.transform(imputer.transform(test_types[TYPE_FEATURES]))
        TYPE_ASSIGNMENTS=test_types[['oktmo','region','year']+TYPE_FEATURES].copy()
        TYPE_ASSIGNMENTS['type_id']=cluster.predict(TEST_TYPES)
        profiles=train_types[TYPE_FEATURES].copy();profiles['type_id']=cluster.labels_
        TYPE_REPORT=(profiles.groupby('type_id').agg(n=('type_id','count'),
             change_1yr=('pop_growth_lag1','median'),natural=('natural_rate_lag1','median'),
             migration=('migration_rate_lag1','median')).reset_index())
        TYPE_REPORT['type_name']=TYPE_REPORT.apply(lambda z:
            ('естественная убыль с миграционной компенсацией' if z['natural']<0 and z['migration']>0 else
             'смешанная убыль' if z['natural']<0 and z['migration']<0 else
             'миграционный отток при естественном приросте' if z['natural']>=0 and z['migration']<0 else
             'рост по двум компонентам' if z['natural']>0 and z['migration']>0 else 'смешанный/неопределённый профиль'),axis=1)
        TYPE_REPORT['silhouette_train']=score
        # Блочная региональная устойчивость; центры могут менять индексы типов,
        # поэтому сравниваем ARI по одних и тех же train объектам.
        rng=np.random.default_rng(CFG['SEED']);regions=train_types['region'].unique()
        stability=[]
        for b in range(min(20,5 if CFG['MODE']=='SMOKE' else 20)):
            sampled=rng.choice(regions,len(regions),replace=True)
            ids=np.concatenate([np.flatnonzero(train_types['region'].to_numpy()==s) for s in sampled])
            if len(np.unique(ids))<max(10,chosen_k*5):continue
            kk=KMeans(n_clusters=chosen_k,random_state=CFG['SEED']+b+1,n_init=3).fit(A[ids])
            stability.append({'replicate':b,'ARI_full_train':adjusted_rand_score(cluster.labels_,kk.predict(A))})
        TYPE_STABILITY=pd.DataFrame(stability)
    else: TYPES_READY=False
if not TYPES_READY:
    print('TYPOLOGY NOT_FEASIBLE: недостаточная история для воспроизводимой кластеризации')
tick('Типы траекторий')


## 11. Пространственная переносимость и контрольная статистика

Независимые территориальные группы формируются **по регионам**. В `TERRITORY_ONLY` тестовые регионы исключены из обучения, годы теста не выходят за TRAIN; в `CROSSED` регионы также новые, а тестовые годы позднее DEV. Эти протоколы не считаются заменой главному `ROLLING_TIME`. Сравнения выполняются только на идентичных ключах, а неопределённость оценивается через повторную выборку регионов, не отдельных соседних строк.

In [ ]:
GROUP_RESULTS=[];GROUP_PREDS=[];GROUP_STATUS='NOT_FEASIBLE'
REGIONS=np.sort(panel['region'].dropna().unique())
if len(REGIONS)>=12 and VALID_H:
    rng=np.random.default_rng(CFG['SEED'])
    held=set(rng.choice(REGIONS,max(3,int(.2*len(REGIONS))),replace=False).tolist())
    for h in VALID_H:
        z=labels[h].copy();complete=z['P_real'].notna()&z['y_delta_rate'].notna()
        train=z[complete&(~z['region'].isin(held))&(z['target_year']<=TRAIN_CUTOFF)].copy()
        if len(train)<CFG['MIN_TRAIN_ROWS']:continue
        x=regression_model('hgb').fit(train[FEATURES],train['y_delta_rate'])
        for protocol in ['TERRITORY_ONLY','CROSSED']:
            if protocol=='TERRITORY_ONLY':
                test=z[complete & z['region'].isin(held) & (z['target_year']<=TRAIN_CUTOFF)].copy()
            else:
                test=z[complete & z['region'].isin(held) & (z['target_year']>DEV_CUTOFF)].copy()
            if len(test)<CFG['MIN_TEST_ROWS']:continue
            assert not set(train['region']) & set(test['region'])
            pred=np.maximum(0,test['population']*(1+x.predict(test[FEATURES])/1000))
            metr=metric_population(test['P_real'],pred,test['population'])
            GROUP_RESULTS.append({'protocol':protocol,'horizon':h,'held_regions':len(held),**metr})
            d=test[['oktmo','region','year','target_year','P_real']].copy()
            d['P_prediction']=pred;d['protocol']=protocol;d['horizon']=h
            GROUP_PREDS.append(d.rename(columns={'year':'origin_year'}))
    if GROUP_RESULTS:GROUP_STATUS='PASS'
GROUP_RESULTS=pd.DataFrame(GROUP_RESULTS)
GROUP_PREDS=pd.concat(GROUP_PREDS,ignore_index=True) if GROUP_PREDS else pd.DataFrame()

# Отбор в TEST не зависит от ошибок и одинаков для всех основных моделей.
assert PRED.empty or not PRED.duplicated(['oktmo','origin_year','horizon','split_id']).any()
STAT_ROWS=[];INTERVAL_ROWS=[];COMPONENT_ROWS=[];PAIRED_ROWS=[]
if not PRED.empty:
    nboot=CFG['SMOKE_BOOTSTRAP'] if CFG['MODE']=='SMOKE' else CFG['N_BOOTSTRAP']
    rng=np.random.default_rng(CFG['SEED'])
    for h,grp in PRED.groupby('horizon'):
        # Проверка одинаковых ключей и определений P.
        assert grp[['P_real','P_component','P_direct','P_last']].notna().all().all()
        assert grp['target_year'].eq(grp['origin_year']+h).all()
        assert (grp['information_cutoff']<=grp['origin_year']).all()
        assert grp['comparable_flag'].all()
        for x in ['B','D','M']:
            err=grp[f'{x}_pred']-grp[f'{x}_real']
            COMPONENT_ROWS.append({'horizon':h,'component':x,'N':len(err),
                 'MAE':float(err.abs().mean()),'bias':float(err.mean()),
                 'RMSE':float(np.sqrt(np.mean(err**2)))})
        # Bootstrap по регионам: сохраняет внутри кластеров корреляцию муниципалитетов и лет.
        region_groups={reg:g for reg,g in grp.groupby('region')}
        regions=list(region_groups)
        comparisons=[('H2.1','COMPONENT_HGB','P_component','DIRECT_HGB','P_direct'),
                     ('H2.1_baseline','DIRECT_HGB','P_direct','LAST_P','P_last')]
        for hyp,modelA,colA,modelB,colB in comparisons:
            errorA=(grp[colA]-grp['P_real']).abs().to_numpy()
            errorB=(grp[colB]-grp['P_real']).abs().to_numpy()
            delta=float(np.mean(errorA-errorB))  # отрицательное значение: A лучше B
            diffs=[]
            for _ in range(nboot):
                sampled=rng.choice(regions,size=len(regions),replace=True)
                # Взвешивание по размеру выбранного кластера, не по числу строк IID.
                values=np.concatenate([(g[colA]-g['P_real']).abs().to_numpy()-(g[colB]-g['P_real']).abs().to_numpy()
                                       for r0 in sampled for g in [region_groups[r0]]])
                diffs.append(np.mean(values))
            lo,hi=np.quantile(diffs,[.025,.975]);p=2*min((np.sum(np.array(diffs)<=0)+1)/(len(diffs)+1),(np.sum(np.array(diffs)>=0)+1)/(len(diffs)+1))
            STAT_ROWS.append({'hypothesis':hyp,'horizon':h,'A':modelA,'B':modelB,
              'delta_MAE_A_minus_B':delta,'ci95_low':float(lo),'ci95_high':float(hi),
              'p_boot_raw':min(1.0,float(p)),'cluster_unit':'region','bootstrap_reps':nboot})
        # Conformal калибровка по DEV только; TEST не применяется при оценке q.
        dev_abs=np.array([v['abs_error'] for v in DEV_RESIDUALS if v['horizon']==h])
        if len(dev_abs)>=30:
            qrank=min(1.0,math.ceil((len(dev_abs)+1)*.9)/len(dev_abs))
            radius=float(np.quantile(dev_abs,qrank,method='higher'))
            low=np.maximum(0,grp['P_component']-radius);high=grp['P_component']+radius
            cover=grp['P_real'].between(low,high).to_numpy()
            INTERVAL_ROWS.append({'horizon':h,'n_dev':len(dev_abs),'n_test':len(grp),'nominal':.90,
                        'coverage':float(np.mean(cover)),'mean_width':float(np.mean(high-low)),
                        'method':'split-conformal absolute-error; regional/time dependence caveat'})
            PRED.loc[grp.index,'P_component_lower90']=low
            PRED.loc[grp.index,'P_component_upper90']=high
        region_aggregated=(grp.groupby(['region','target_year'],as_index=False)
            .agg(P_real=('P_real','sum'),P_direct=('P_direct','sum'),P_component=('P_component','sum')))
        region_aggregated['region_error_component']=region_aggregated['P_component']-region_aggregated['P_real']
        PAIRED_ROWS.append(region_aggregated.assign(horizon=h))

STATS=pd.DataFrame(STAT_ROWS)
if not STATS.empty:
    # Holm в предварительно зарегистрированном семействе всех прогнозных сравнений.
    order=STATS['p_boot_raw'].sort_values().index.tolist();m=len(order)
    adjusted=pd.Series(np.nan,index=STATS.index)
    previous=0.
    for i,idx in enumerate(order):
        previous=max(previous,min(1.,(m-i)*STATS.loc[idx,'p_boot_raw']))
        adjusted.loc[idx]=previous
    STATS['p_holm']=adjusted
COMP_METRICS=pd.DataFrame(COMPONENT_ROWS)
INTERVALS=pd.DataFrame(INTERVAL_ROWS)
REGIONAL_ERRORS=pd.concat(PAIRED_ROWS,ignore_index=True) if PAIRED_ROWS else pd.DataFrame()
RISK_CALIBRATION=pd.DataFrame()
if not RISK_PREDS.empty:
    bins=np.linspace(0,1,6)
    tmp=RISK_PREDS[['risk_label','prob_RISK_HGB']].copy()
    tmp['bin']=pd.cut(tmp['prob_RISK_HGB'],bins=bins,include_lowest=True)
    RISK_CALIBRATION=tmp.groupby('bin',observed=True).agg(n=('risk_label','size'),
       forecast=('prob_RISK_HGB','mean'),observed=('risk_label','mean')).reset_index()
    RISK_CALIBRATION['bin']=RISK_CALIBRATION['bin'].astype(str)
tick('Статистика · paired cluster bootstrap')


## 12. Интерпретация, включая неподтверждённые результаты

Все утверждения составляются на основе вычисленных метрик. Статистически неопределённая разница не переводится в «успех». Равновесие компонент, достигнутое сложением прогнозов, является математическим свойством агрегирования, но **не гарантирует** верность B/D/M или общей численности. Высокая точность инерционной численности отдельно сопоставляется с ошибкой изменения в расчёте на 1000 человек.

In [ ]:
FINDINGS=[]
if STATS.empty:
    FINDINGS.append('H2.1 NOT_FEASIBLE: недостаточно сопоставимых тестовых рядов P/B/D/M.')
else:
    for _,v in STATS.iterrows():
        verdict=('статистически различаются' if v['p_holm']<.05 else 'статистически не различаются при заданной процедуре')
        FINDINGS.append(f"{v['hypothesis']} h={int(v['horizon'])}: {v['A']} − {v['B']} = {v['delta_MAE_A_minus_B']:.2f} человек MAE, Holm p={v['p_holm']:.3f}; {verdict}.")
if not RISK_METRICS.empty:
    a=RISK_METRICS.loc[(RISK_METRICS['split']=='TEST')&(RISK_METRICS['model']=='RISK_HGB')]
    if len(a):FINDINGS.append(f"H2.2 (только демографические лаги): Brier={a.iloc[0]['Brier']:.4f}, PR-AUC={a.iloc[0]['PR_AUC']:.4f}; влияние экономических факторов не проверялось.")
else:FINDINGS.append('H2.2 NOT_FEASIBLE: нет корректно разделённых рядов двух классов риска.')
if TYPES_READY and isinstance(TYPE_REPORT,pd.DataFrame):
    med=float(TYPE_STABILITY['ARI_full_train'].median()) if not TYPE_STABILITY.empty else np.nan
    FINDINGS.append(f'H2.3: {len(TYPE_REPORT)} исторических типов, медиана ARI bootstrap={med:.3f}; это описательная структура, не доказательство причин.')
else:FINDINGS.append('H2.3 NOT_FEASIBLE: недостаточно наблюдений для типологии.')
if not INTERVALS.empty:
    FINDINGS.append('H2.4: фактическое покрытие 90%-х интервалов: '+', '.join(f"h={int(v.horizon)}: {v.coverage:.1%}" for v in INTERVALS.itertuples())+'.')
else:FINDINGS.append('H2.4 NOT_FEASIBLE: для независимой калибровки не хватает DEV.')
FINDINGS.append('Отсутствует независимое сопоставление демографического раздела БДМО: такого источника в доступном handoff нет.' if not BDMO_DEMOG else 'Демографический раздел БДМО обнаружен в handoff; самостоятельная проверка определений требуется до объединения.')
FINDINGS.append('Данные D01 представляют 78 регионов согласно паспорту первоисточника; переход к общероссийским оценкам без весов и охвата недопустим.')
FINDINGS.append('Число строк в индексе Stage I отличается от опубликованного паспорта D01 (31 445 против 31 947); причины следует отдельно проверить по исходному архиву.')
print('\n'.join(FINDINGS[:8]))


## 13. Excel, Parquet, графики, evidence и manifest

В Excel — компактные реальные аналитические таблицы; полная подготовленная панель, прогнозы по каждому муниципалитету и промежуточные модели сохраняются в **едином** `ML2_EVIDENCE.zip`. Если исследование ограничено неполной историей, это прямо отражается в статусах. Архив не является дополнительным независимым источником: он содержит производные от D01.

In [ ]:
EVIDENCE_CONTENT=BASE/'evidence_files';EVIDENCE_CONTENT.mkdir(exist_ok=True)
for old in EVIDENCE_CONTENT.rglob('*'):
    if old.is_file():old.unlink()
(EXCEL_PATH,ZIP_PATH,MANIFEST_PATH,NB_PATH)=(
    OUT/'ML2_DEMOGRAPHY_RESULTS.xlsx',OUT/'ML2_EVIDENCE.zip',OUT/'run_manifest.json',OUT/'ML2_DEMOGRAPHY.ipynb')

# Информация о реальных компонентах и источнике не теряется при построении панели.
panel.to_parquet(EVIDENCE_CONTENT/'municipal_panel.parquet',index=False)
origins[['oktmo','region','year']+FEATURES].to_parquet(EVIDENCE_CONTENT/'historical_features.parquet',index=False)
if not PRED.empty:PRED.to_parquet(EVIDENCE_CONTENT/'forecasts.parquet',index=False)
if not PRED_LONG.empty:PRED_LONG.to_parquet(EVIDENCE_CONTENT/'forecasts_long.parquet',index=False)
if not TRAJECTORIES.empty:TRAJECTORIES.to_parquet(EVIDENCE_CONTENT/'annual_BDM_trajectories.parquet',index=False)
if not RISK_PREDS.empty:RISK_PREDS.to_parquet(EVIDENCE_CONTENT/'risk_predictions.parquet',index=False)
if not TYPE_ASSIGNMENTS.empty:TYPE_ASSIGNMENTS.to_parquet(EVIDENCE_CONTENT/'type_assignments.parquet',index=False)
if not GROUP_PREDS.empty:GROUP_PREDS.to_parquet(EVIDENCE_CONTENT/'group_holdouts.parquet',index=False)
if not METRICS.empty:METRICS.to_parquet(EVIDENCE_CONTENT/'model_metrics.parquet',index=False)
with open(EVIDENCE_CONTENT/'models.joblib','wb') as f:joblib.dump(MODELS,f,compress=3)
(EVIDENCE_CONTENT/'experiment_design.json').write_text(json.dumps(DESIGN,ensure_ascii=False,indent=2),encoding='utf-8')
(EVIDENCE_CONTENT/'source_provenance.json').write_text(json.dumps({
    'repo':CFG['REPO'],'ref_sha':HEAD_SHA,'asset':D01_ASSET,'index_sha256':hashlib.sha256(INDEX_BYTES).hexdigest(),
    'license_primary_D01':'CC BY 4.0','original_data_note':'Population retrospectively reconstructed for temporal comparability',
    'fields_with_lag':FEATURES,'source_years':[int(panel.year.min()),int(panel.year.max())]},
    ensure_ascii=False,default=str,indent=2),encoding='utf-8')

PLOT_FILES=[]
plt.rcParams.update({'font.family':'DejaVu Sans','figure.dpi':125,'axes.grid':True,'grid.alpha':.18})
def save_figure(name):
    path=EVIDENCE_CONTENT/(name+'.png')
    plt.savefig(path,bbox_inches='tight',dpi=160);plt.close();PLOT_FILES.append(path)

if not METRICS.empty:
    z=METRICS.loc[(METRICS['split']=='TEST')&(METRICS['model'].isin(['DIRECT_HGB','COMPONENT_HGB','LAST_P']))]
    fig,ax=plt.subplots(figsize=(9,4.8))
    for name,d in z.groupby('model'):
        ax.plot(d['horizon'],d['MAE_P'],marker='o',label=name)
    ax.set(xlabel='Горизонт, лет',ylabel='MAE численности, человек',title='Ошибка прогноза по горизонту');ax.legend()
    save_figure('01_accuracy_horizons')
if not COMP_METRICS.empty:
    p=COMP_METRICS.pivot(index='horizon',columns='component',values='MAE')
    ax=p.plot(kind='bar',figsize=(8.8,4.5),title='Абсолютная ошибка компонент (накопленная за горизонт)',rot=0)
    ax.set(xlabel='Горизонт, лет',ylabel='MAE, человек');plt.tight_layout();save_figure('02_BDM_errors')
if not qa_balance.empty:
    fig,ax=plt.subplots(figsize=(9,4.2))
    ax.bar(qa_balance['year'].astype(str),qa_balance['mean_abs_residual'])
    ax.tick_params(axis='x',labelrotation=45)
    ax.set(xlabel='Год',ylabel='Средний абсолютный остаток, человек',title='Балансовый остаток D01 (после реконструкции ряда)')
    save_figure('03_balance_residual')
if not RISK_CALIBRATION.empty:
    fig,ax=plt.subplots(figsize=(5.8,5.2))
    ax.plot([0,1],[0,1],ls='--',label='Идеальная калибровка')
    ax.scatter(RISK_CALIBRATION['forecast'],RISK_CALIBRATION['observed'],
               s=RISK_CALIBRATION['n'].clip(lower=10).to_numpy()/5,label='RISK-HGB')
    ax.set(xlim=(0,1),ylim=(0,1),xlabel='Прогнозируемая вероятность',ylabel='Наблюдаемая доля',title='Калибровка трёхлетнего риска');ax.legend()
    save_figure('04_risk_calibration')
if TYPES_READY and isinstance(TYPE_REPORT,pd.DataFrame) and not TYPE_REPORT.empty:
    fig,ax=plt.subplots(figsize=(8,4.2))
    ax.scatter(TYPE_REPORT['natural'],TYPE_REPORT['migration'],s=TYPE_REPORT['n'].clip(lower=10)/6)
    for row in TYPE_REPORT.itertuples():ax.annotate(str(row.type_id),(row.natural,row.migration),xytext=(3,3),textcoords='offset points')
    ax.axhline(0,color='grey',lw=.7);ax.axvline(0,color='grey',lw=.7)
    ax.set(xlabel='Медианная естественная динамика, на 1000',ylabel='Медианное миграционное сальдо, на 1000',title='Эмпирические типы прошлых траекторий')
    save_figure('05_typology_components')
if not INTERVALS.empty:
    fig,ax=plt.subplots(figsize=(7.5,4.2))
    ax.plot(INTERVALS['horizon'],INTERVALS['coverage'],marker='o',label='Фактическое покрытие')
    ax.axhline(.9,color='grey',ls='--',label='Номинал 90%')
    ax.set(xlabel='Горизонт, лет',ylabel='Доля попаданий',ylim=(0,1.03),title='Проверка интервалов на отложенном времени');ax.legend()
    save_figure('06_interval_coverage')
assert len(PLOT_FILES)<=CFG['MAX_PLOTS']

statuses={
 'PASS_INPUT':True,'PASS_DATA':len(panel)>0,'PASS_SPLITS':bool(VALID_H) and bool(assert_future_canary(panel)),
 'PASS_TRAINING':bool(VALID_H) and len(MODELS)>0 and not PRED.empty,
 'PASS_VALIDATION':not STATS.empty and not COMP_METRICS.empty,
 'PASS_INTERPRETATION':len(FINDINGS)>=4,
 'PASS_OUTPUTS':False,'PASS_NOTEBOOK':False,'PASS_GITHUB':False,
 'RISK':'PASS' if RISK_READY else 'NOT_FEASIBLE',
 'TERRITORY_AND_CROSSED':GROUP_STATUS,
 'TYPOLOGY':'PASS' if TYPES_READY else 'NOT_FEASIBLE',
 'BDMO_DEMOGRAPHY':'NOT_FEASIBLE' if not BDMO_DEMOG else 'PRESENT_NOT_MERGED',
}
# Отсутствие необязательных протоколов не меняет критические требования ML-2.
LOCAL_RESEARCH_PASS=all(statuses[k] is True for k in
   ['PASS_INPUT','PASS_DATA','PASS_SPLITS','PASS_TRAINING','PASS_VALIDATION','PASS_INTERPRETATION'])
status_label=('FULL_COMPUTED_PENDING_QA' if RUN_FULL and LOCAL_RESEARCH_PASS else
              'SMOKE_ONLY' if not RUN_FULL else 'PARTIAL_NOT_FEASIBLE')

pass_list=pd.DataFrame([{'Контроль':k,'Статус':('PASS' if v is True else 'FAIL' if v is False else v)}
                        for k,v in statuses.items()])
main_pass=pd.DataFrame([
 {'Раздел':'Идентификатор','Значение':'ML-02 Демографическая устойчивость'},
 {'Раздел':'Статус при подготовке Excel (итог в manifest)','Значение':status_label},
 {'Раздел':'Версия','Значение':CFG['VERSION']},
 {'Раздел':'Репозиторий','Значение':CFG['REPO']},
 {'Раздел':'Git source commit','Значение':HEAD_SHA},
 {'Раздел':'Наблюдений панели','Значение':len(panel)},
 {'Раздел':'Муниципалитетов','Значение':panel['oktmo'].nunique()},
 {'Раздел':'Годы','Значение':f"{panel.year.min()}—{panel.year.max()}"},
 {'Раздел':'Горизонты с результатом','Значение':','.join(map(str,VALID_H)) or 'нет'},
 {'Раздел':'Сделано прогнозов','Значение':len(PRED)},
 {'Раздел':'Модельных объектов','Значение':len(MODELS)},
 {'Раздел':'Сравнений bootstrap','Значение':len(STATS)},
 {'Раздел':'Источник','Значение':'D01: Росстат / Если быть точным (2024), CC BY 4.0'},
 {'Раздел':'Ключевая оговорка','Значение':'P на 1 января; D01 содержит реконструированную историю'},
])
findings_table=pd.DataFrame({'Научное заключение':FINDINGS})

sheets=[('Паспорт',main_pass),('Контроль',pass_list),('Охват_по_годам',qa_years),('DATA_GATE',DATA_PLAN),
        ('Ключи_полнота',qa_key),('Баланс_населения',qa_balance),
        ('Компоненты_BDM',COMP_METRICS),('Прямой_компонентный',METRICS.loc[METRICS['split']=='TEST'] if not METRICS.empty else METRICS),
        ('Риски_депопуляции',RISK_METRICS),('Калибровка_риска',RISK_CALIBRATION),
        ('Типы_траекторий',TYPE_REPORT if isinstance(TYPE_REPORT,pd.DataFrame) else pd.DataFrame()),
        ('Устойчивость_типов',TYPE_STABILITY),('Интервалы_прогноза',INTERVALS),
        ('Группы_муниципалитетов',GROUP_RESULTS),('Проверка_гипотез',STATS),
        ('Региональные_ошибки',REGIONAL_ERRORS),('Выводы',findings_table)]
with pd.ExcelWriter(EXCEL_PATH,engine='openpyxl') as writer:
    for name,data in sheets:
        if isinstance(data,pd.DataFrame) and not data.empty:
            data.to_excel(writer,sheet_name=name,index=False)
wb=load_workbook(EXCEL_PATH)
for ws in wb.worksheets:
    ws.freeze_panes='A2';ws.auto_filter.ref=ws.dimensions
    for cell in ws[1]:cell.font=Font(bold=True,color='FFFFFF');cell.fill=PatternFill('solid',fgColor='20334A')
    for cells in ws.columns:
        letter=get_column_letter(cells[0].column)
        width=max((min(70,len(str(c.value or ''))) for c in list(cells)[:400]),default=12)
        ws.column_dimensions[letter].width=max(13,min(63,width+3))
wb.save(EXCEL_PATH)
tick('Экспорт · Excel + панели')

# Один технический архив: модельные состояния, исследовательская панель, прогнозы, QA и изображения.
(EVIDENCE_CONTENT/'run_log.txt').write_text(
    f"ML-02 {CFG['VERSION']} mode={CFG['MODE']}\nDATA rows={len(panel)} years={panel.year.min()}..{panel.year.max()}\n"+
    f"models={len(MODELS)} forecasts={len(PRED)} design_sha={DESIGN_SHA}\n"+'\n'.join(FINDINGS),encoding='utf-8')
with ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
    for path in sorted(EVIDENCE_CONTENT.iterdir()):archive.write(path,arcname=path.name)
MANIFEST={
 'experiment':'ML-02', 'version':CFG['VERSION'],'mode':CFG['MODE'],
 'environment':resources,
 'status':status_label, 'created_at':dt.datetime.now(dt.timezone.utc).isoformat(),
 'repository':CFG['REPO'],'source_ref':CFG['SOURCE_REF'],'source_commit':HEAD_SHA,
 'input_index_sha256':hashlib.sha256(INDEX_BYTES).hexdigest(),
 'input_assets':{D01_ASSET['asset_id']:D01_ASSET['sha256']},
 'data':{'municipalities':int(panel.oktmo.nunique()),'rows':len(panel),
         'years':[int(panel.year.min()),int(panel.year.max())],'valid_horizons':VALID_H},
 'run':{'models':len(MODELS),'forecasts':len(PRED),'bootstrap_comparisons':len(STATS),
        'elapsed_sec':round(time.monotonic()-START,1)},
 'research_design_sha256':DESIGN_SHA, 'checks':statuses,
 'outputs':{'ML2_DEMOGRAPHY_RESULTS.xlsx':sha256_file(EXCEL_PATH),
            'ML2_EVIDENCE.zip':sha256_file(ZIP_PATH)},
 'publication':{'status':'NOT_STARTED','branch':None,'tag':None,'commit_sha':None,
                'links':{}}
}
MANIFEST_PATH.write_text(json.dumps(MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
tick('Экспорт · evidence + manifest')
print('Подготовлены локальные XLSX/ZIP; научный статус:',status_label)


## 14. Локальный QA, проверка ноутбука и выпуск в приватный GitHub

Публикация разрешена **только** в режиме FULL после `PASS_INPUT`, `PASS_DATA`, `PASS_SPLITS`, `PASS_TRAINING`, `PASS_VALIDATION`, `PASS_INTERPRETATION`, `PASS_OUTPUTS`, `PASS_NOTEBOOK`. `SMOKE` никогда не публикует релиз. GitHub получает новый branch и versioned Release **только после** локальных проверок XLSX/ZIP/Parquet/ноутбука. При сбое статус явно остаётся `PARTIAL`/`PUBLICATION_FAIL`.

In [ ]:
def capture_current_notebook():
    # В Colab запрашивается текущее notebook JSON, которое обычно не находится как файл в /content.
    doc=None
    try:
        from google.colab import _message
        response=_message.blocking_request('get_ipynb',timeout_sec=90)
        if isinstance(response,dict):doc=response.get('ipynb',response)
    except (ImportError,ModuleNotFoundError,AttributeError,KeyError,ValueError,TimeoutError): pass
    if doc is None:
        candidate=Path.cwd()/'ML2_DEMOGRAPHY.ipynb'
        if candidate.is_file():doc=nbformat.read(candidate,as_version=4)
    if doc is None: return None
    notebook=nbformat.from_dict(doc) if isinstance(doc,dict) else doc
    nbformat.validate(notebook)
    for cell in notebook.cells:
        if cell.cell_type=='code':cell.outputs=[];cell.execution_count=None
    notebook.metadata.pop('widgets',None)
    rendered=nbformat.writes(notebook)
    if GH_TOKEN and GH_TOKEN in rendered:raise ValueError('Секрет обнаружен в notebook JSON')
    if re.search(r'gh[pousr]_[A-Za-z0-9_]{25,}|github_pat_[A-Za-z0-9_]{30,}',rendered):
        raise ValueError('Похоже, в notebook JSON присутствует токен')
    if ('%'*2+'writefile') in rendered:raise ValueError('Однофайловый контракт нарушен')
    NB_PATH.write_text(rendered,encoding='utf-8')
    nbformat.validate(nbformat.read(NB_PATH,as_version=4))
    return NB_PATH

qa_errors=[]
try:
    wb_check=load_workbook(EXCEL_PATH,read_only=True,data_only=True)
    if not wb_check.sheetnames:qa_errors.append('Пустой Excel')
    for s in wb_check:
        if s.max_row<2 or s.max_column<1:qa_errors.append('Пустой лист: '+s.title)
    wb_check.close()
    with ZipFile(ZIP_PATH,'r') as z:
        if z.testzip() is not None:qa_errors.append('ZIP CRC FAIL')
        listed=z.namelist()
        required={'municipal_panel.parquet','historical_features.parquet','models.joblib','experiment_design.json','source_provenance.json'}
        if not required.issubset(set(listed)):qa_errors.append('Не хватает файлов evidence')
        if RUN_FULL and VALID_H and 'forecasts.parquet' not in listed:qa_errors.append('Нет реальных прогнозов')
        # Каждая Parquet таблица действительно читается.
        for x in listed:
            if x.endswith('.parquet'):
                meta=pq.ParquetFile(pa.BufferReader(z.read(x))).metadata
                if meta.num_rows<1:qa_errors.append('Пустой Parquet: '+x)
    if not PRED.empty:
        assert not PRED.duplicated(['oktmo','origin_year','horizon','split_id']).any()
        if (PRED[['P_direct','P_component','P_last']]<0).any().any():qa_errors.append('Отрицательное P в прогнозах')
        if (PRED[['B_pred','D_pred']]<0).any().any():qa_errors.append('Отрицательные B или D')
        if (PRED['balance_error_forecast'].abs()>1e-6).any():qa_errors.append('COMPONENT нарушает тождество вследствие ограничения P')
        if not (PRED['target_year']==PRED['origin_year']+PRED['horizon']).all():qa_errors.append('Несогласованные временные ключи')
    if sha256_file(D01_FILE)!=D01_ASSET['sha256']:qa_errors.append('D01 checksum изменился')
except Exception as exc:
    qa_errors.append('Ошибка QA: '+type(exc).__name__)
try:
    capture_current_notebook()
except Exception as exc:
    qa_errors.append('Не удалось безопасно извлечь ipynb: '+type(exc).__name__)
statuses['PASS_OUTPUTS']=not any(x for x in qa_errors if 'ipynb' not in x and 'ноутбук' not in x)
statuses['PASS_NOTEBOOK']=NB_PATH.is_file() and nbformat.validate(nbformat.read(NB_PATH,as_version=4)) is None
MANIFEST['checks']=statuses
if NB_PATH.is_file():MANIFEST['outputs']['ML2_DEMOGRAPHY.ipynb']=sha256_file(NB_PATH)
MANIFEST['status']=('FULL_LOCAL_QA_PASS' if RUN_FULL and LOCAL_RESEARCH_PASS and statuses['PASS_OUTPUTS'] and statuses['PASS_NOTEBOOK']
                    else 'SMOKE_ONLY' if not RUN_FULL else 'PARTIAL_QA_OR_SCIENCE')
MANIFEST['qa_errors']=qa_errors
MANIFEST['run']['elapsed_sec']=round(time.monotonic()-START,1)
MANIFEST_PATH.write_text(json.dumps(MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
tick('Локальный QA · notebook / Excel / ZIP')
print('Локальный QA:',MANIFEST['status'],'·',('нет ошибок' if not qa_errors else '; '.join(qa_errors[:4])))


### Контролируемая публикация

Ветка создаётся от исходного `main` в момент публикации и используется только для `stage2/ml02_demography`. Для технического архива создаётся GitHub Release с уникальным тегом; загрузка SHA-256 проверяется повторным скачиванием. Если ноутбук запущен без доступного самоэкспорта текущего `.ipynb`, научные XLSX/ZIP остаются локально, а публикация **не** проводится — это проверяемая блокировка, а не ложный `PASS`.

In [ ]:
def gh_post(url,body):
    response=requests.post(url,headers=GH_HEADERS,json=body,timeout=(30,180))
    gh_error(response,'GitHub POST');return response.json()
def put_contents(path,source,branch):
    f=Path(source)
    body={'message':'ML-02 verified experiment: '+f.name,
          'branch':branch,'content':base64.b64encode(f.read_bytes()).decode('ascii')}
    response=requests.put(GH_API+'/contents/'+quote(path,safe='/'),headers=GH_HEADERS,json=body,timeout=(30,900))
    gh_error(response,'GitHub PUT '+f.name)
    return response.json()['commit']['sha']
def gh_download_asset(asset_id,path):
    uri=GH_API+'/releases/assets/'+str(asset_id)
    resp=requests.get(uri,headers={**GH_HEADERS,'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
    gh_error(resp,'GitHub asset readback')
    if 'json' in resp.headers.get('Content-Type','').lower():
        resp.close();obj=gh_get_json(uri)
        resp=requests.get(obj['browser_download_url'],headers=GH_HEADERS,stream=True,timeout=(30,900))
        gh_error(resp,'GitHub asset readback redirect')
    with open(path,'wb') as f:
        for chunk in resp.iter_content(4*1024**2):
            if chunk:f.write(chunk)
    resp.close();return Path(path)

def gh_verify_file(repo_path,branch,reference_path):
    remote=gh_content(repo_path,ref=branch)
    return hashlib.sha256(remote).hexdigest()==sha256_file(reference_path)

def publish_verified():
    # Не вызывается из SMOKE и не вызывается при любом критическом FAIL.
    must=['PASS_INPUT','PASS_DATA','PASS_SPLITS','PASS_TRAINING',
          'PASS_VALIDATION','PASS_INTERPRETATION','PASS_OUTPUTS','PASS_NOTEBOOK']
    if not (RUN_FULL and CFG['AUTO_PUBLISH'] and all(statuses[k] is True for k in must)):
        return {'status':'NOT_STARTED','reason':'SMOKE / AUTO_PUBLISH=False / не пройден критический QA'}
    stem=dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    branch=CFG['BRANCH_PREFIX']+'/'+stem
    gh_post(GH_API+'/git/refs',{'ref':'refs/heads/'+branch,'sha':HEAD_SHA})
    segment=CFG['OUTPUT_SEGMENT']
    commit_nb=put_contents(segment+'/ML2_DEMOGRAPHY.ipynb',NB_PATH,branch)
    if EXCEL_PATH.stat().st_size < CFG['RELEASE_ASSET_THRESHOLD_MB']*1024**2:
        commit_excel=put_contents(segment+'/ML2_DEMOGRAPHY_RESULTS.xlsx',EXCEL_PATH,branch)
    else:commit_excel=None
    tag=CFG['VERSION']+'-'+stem
    release=gh_post(GH_API+'/releases',{'tag_name':tag,'target_commitish':branch,
        'name':'ML-02 · Демографическая устойчивость · '+tag,
        'body':'Результат FULL после QA; источник Stage I; исследовательская панель — в ML2_EVIDENCE.zip',
        'draft':False,'prerelease':True})
    upload_base='https://uploads.github.com/repos/'+CFG['REPO']+'/releases/'+str(release['id'])+'/assets'
    uploaded={}
    to_upload=[ZIP_PATH]+([EXCEL_PATH] if commit_excel is None else [])
    for file in to_upload:
        resp=requests.post(upload_base,params={'name':file.name},
          headers={**GH_HEADERS,'Content-Type':'application/octet-stream'},
          data=file.read_bytes(),timeout=(30,900))
        gh_error(resp,'GitHub Release asset upload');uploaded[file.name]=resp.json()
    # Readback полноценно сверяет Git blobs и каждое Release asset.
    if not gh_verify_file(segment+'/ML2_DEMOGRAPHY.ipynb',branch,NB_PATH):
        raise RuntimeError('PUBLICATION_FAIL: readback ipynb SHA mismatch')
    if commit_excel is not None and not gh_verify_file(segment+'/'+EXCEL_PATH.name,branch,EXCEL_PATH):
        raise RuntimeError('PUBLICATION_FAIL: readback XLSX SHA mismatch')
    for f in to_upload:
        copy=BASE/('readback_'+f.name)
        gh_download_asset(uploaded[f.name]['id'],copy)
        if sha256_file(copy)!=sha256_file(f):raise RuntimeError('PUBLICATION_FAIL: readback '+f.name)
        if f==ZIP_PATH:
            with ZipFile(copy) as z:
                if z.testzip():raise RuntimeError('PUBLICATION_FAIL: asset archive damaged')
        copy.unlink(missing_ok=True)
    # Manifest публикуется последним; содержит SHA коммитов контента, которым принадлежит этот отчёт.
    links={
       'notebook':f"https://github.com/{CFG['REPO']}/blob/{branch}/{segment}/{NB_PATH.name}",
       'excel':(f"https://github.com/{CFG['REPO']}/blob/{branch}/{segment}/{EXCEL_PATH.name}"
                if commit_excel is not None else uploaded[EXCEL_PATH.name]['browser_download_url']),
       'evidence':uploaded[ZIP_PATH.name]['browser_download_url'],
       'release':release['html_url']}
    statuses['PASS_GITHUB']=True
    MANIFEST['status']='FULL_PASS'
    MANIFEST['checks']=statuses
    MANIFEST['publication']={'status':'VERIFIED','branch':branch,'tag':tag,
       'commit_sha':commit_excel or commit_nb,'links':links}
    MANIFEST['run']['elapsed_sec']=round(time.monotonic()-START,1)
    MANIFEST_PATH.write_text(json.dumps(MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    commit_manifest=put_contents(segment+'/run_manifest.json',MANIFEST_PATH,branch)
    if not gh_verify_file(segment+'/run_manifest.json',branch,MANIFEST_PATH):
        raise RuntimeError('PUBLICATION_FAIL: manifest readback SHA mismatch')
    return {'status':'VERIFIED','branch':branch,'tag':tag,'commit':commit_manifest,'links':links}

publication={'status':'NOT_STARTED'}
try:
    publication=publish_verified()
except Exception as exc:
    statuses['PASS_GITHUB']=False
    publication={'status':'PUBLICATION_FAIL','reason':type(exc).__name__}
    MANIFEST['status']='PUBLICATION_FAIL'
    MANIFEST['checks']=statuses
    MANIFEST['publication']=publication
    MANIFEST_PATH.write_text(json.dumps(MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
tick('GitHub · '+publication.get('status','UNKNOWN'))
MAIN_BAR.close()
print('Публикация:',publication.get('status','UNKNOWN'))


## 15. Итог и статус научной приёмки

Ниже — исключительно **фактически вычисленные** результаты. Статус `VERIFIED` присваивается только после повторного скачивания четырёх публикационных объектов с проверкой SHA. `SMOKE`, `NOT_FEASIBLE`, `BLOCKED` и `PUBLICATION_FAIL` не приравниваются к завершённому ML-2.

In [ ]:
if not publication.get('status')=='VERIFIED' and MANIFEST['status']!='PUBLICATION_FAIL':
    MANIFEST['status']=('SMOKE_ONLY' if CFG['MODE']=='SMOKE' else
                        'FULL_LOCAL_QA_PASS_NOT_PUBLISHED' if all(statuses.get(k) is True for k in
                           ['PASS_INPUT','PASS_DATA','PASS_SPLITS','PASS_TRAINING','PASS_VALIDATION',
                            'PASS_INTERPRETATION','PASS_OUTPUTS','PASS_NOTEBOOK']) else 'PARTIAL_NOT_PUBLISHED')
    MANIFEST['publication']=publication
    MANIFEST['checks']=statuses
    MANIFEST_PATH.write_text(json.dumps(MANIFEST,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
summary={
 'status':MANIFEST['status'],'municipalities':int(panel.oktmo.nunique()),'observed_years':f"{panel.year.min()}–{panel.year.max()}",
 'predictions':len(PRED),'trained_models':len(MODELS),
 'evaluated_horizons':VALID_H,'risk_status':statuses['RISK'],
 'spatial_status':statuses['TERRITORY_AND_CROSSED'],
 'output_dir':str(OUT),'git_release':publication.get('links',{}).get('release','not published')}
print(json.dumps(summary,ensure_ascii=False,indent=2))
if not METRICS.empty:
    print(METRICS.loc[(METRICS['split']=='TEST') & (METRICS['model'].isin(['DIRECT_HGB','COMPONENT_HGB','LAST_P'])),
                      ['horizon','model','N','MAE_P','MAE_change_per1000']].round(3).to_string(index=False))
if publication.get('status')!='VERIFIED':
    print('Только локальная выгрузка. В GitHub НЕ опубликовано:',publication.get('reason','не пройден FULL QA или публикация отключена'))
